In [ ]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "archive/src/experiment.py").is_file()
)
PROJECT_ROOT = ROOT = REPOSITORY_ROOT / "archive"
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))


# CNN: сравнение аугментаций на 256×256
Выберите один CONFIG и новый EXP_ID, затем запустите ячейки сверху вниз.
Все четыре варианта имеют одинаковую CNN и параметры обучения; меняется только augmentation.
Начните с no_aug, затем geometry, photo и domain_shift. Не запускайте все одновременно.
Val не аугментируется; test не используется. Raw не меняется. Логи, графики, best/last checkpoints,
предсказания val и строка experiments.csv создаются общим pipeline.
Кривые train измеряются с аугментациями и dropout: основное сравнение — Accuracy на исходной val.
Описание диапазонов: artifacts/reviews/augmentations.md. Сильные искажения не гарантируют улучшения.

In [ ]:
from pathlib import Path
import sys


from archive.src.experiment import load_config, run_experiment
from archive.src.transforms import preview_augmentations
from archive.src.utils import get_device
from archive.src.data import ImageDataset
import matplotlib.pyplot as plt


In [ ]:
CONFIG = "configs/cnn_256_domain_shift.json"
# CONFIG = "configs/cnn_256_no_aug.json"
# CONFIG = "configs/cnn_256_geometry.json"
# CONFIG = "configs/cnn_256_photo.json"

EXP_ID = "cnn_256_domain_shift_s42_001"  # Новый уникальный ID на каждый запуск
DEVICE = "auto"  # auto, cpu, cuda
config = load_config(CONFIG, project_root=PROJECT_ROOT)
config["training"]["device"] = DEVICE
# На Kaggle при необходимости укажите каталог с train.csv, val.csv, classes.csv и train/val/:
# config["data"]["raw_dir"] = "/kaggle/input/<dataset>"
print(f"Конфигурация: {CONFIG}; устройство: {get_device(DEVICE)}")
print(config["data"]["augmentation"])


In [ ]:
# Необязательный просмотр только одного train-изображения; RNG обучения не меняется.
raw = Path(config["data"]["raw_dir"])
if not raw.is_absolute():
    raw = PROJECT_ROOT / raw
image, _ = ImageDataset(raw / "train.csv", raw / "train")[0]
fig = preview_augmentations(image, config["data"], seed=config["seed"])
plt.show()
plt.close(fig)


In [ ]:
result = run_experiment(config, exp_id=EXP_ID, project_root=PROJECT_ROOT)
